In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Epithelial reclustering + R/NR composition + DE (cluster vs rest) using Scanpy.

Input:
  /ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/01_mapping_raw_scRNA_seq_to_reference/adata_scvi_integrated_all_cells.h5ad

Output dir:
  /ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/08_other_cell_types/epithelial/epithelial_cells

Steps:
  1) Subset predicted_labels == "Epithelial cells"
  2) Neighbors/UMAP/Leiden (res=0.6) using X_scvi latent
  3) Count Non-Responder vs Responder per cluster (absolute cell counts) + barplots
  4) DE: leiden cluster vs rest using layer norm_expr_log1p, save per-cluster signature genes
"""

import os
import warnings
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

warnings.filterwarnings("ignore")

# ----------------------------
# Matplotlib config (PDF friendly)
# ----------------------------
mpl.rcParams["font.family"] = "Liberation Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ----------------------------
# Paths
# ----------------------------
H5AD_PATH = (
    "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/"
    "01_mapping_raw_scRNA_seq_to_reference/adata_scvi_integrated_all_cells.h5ad"
)

OUTDIR = (
    "/ShangGaoAIProjects/Zang/single_cell_data/data_analysis_pipeline_simplified/"
    "08_other_cell_types/epithelial/epithelial"
)

FIGDIR = os.path.join(OUTDIR, "figures")
TABDIR = os.path.join(OUTDIR, "tables")
DEDIR  = os.path.join(OUTDIR, "de")
SIGDIR = os.path.join(OUTDIR, "signatures")
for d in [OUTDIR, FIGDIR, TABDIR, DEDIR, SIGDIR]:
    os.makedirs(d, exist_ok=True)

sc.settings.verbosity = 2

# ----------------------------
# Helpers
# ----------------------------
def pick_obs_col(adata, candidates):
    lower_map = {c.lower(): c for c in adata.obs.columns}
    for cand in candidates:
        if cand in adata.obs.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def pick_scvi_rep(adata):
    for k in ["X_scVI", "X_scvi", "X_scanvi", "X_scANVI"]:
        if k in adata.obsm:
            return k
    for k in adata.obsm.keys():
        lk = k.lower()
        if "scvi" in lk or "scanvi" in lk:
            return k
    return None

def save_current_fig(path_png, path_pdf=None, dpi=220):
    plt.tight_layout()
    plt.savefig(path_png, dpi=dpi)
    if path_pdf is not None:
        plt.savefig(path_pdf)
    plt.close()

# ----------------------------
# Load
# ----------------------------
print(f"[1/6] Loading: {H5AD_PATH}")
ad = sc.read_h5ad(H5AD_PATH)

# ----------------------------
# Subset epithelial
# ----------------------------
print("[2/6] Subsetting predicted_labels == 'Epithelial cells'")
label_col = pick_obs_col(ad, ["predicted_labels", "predicted_label", "celltype", "cell_type"])
if label_col is None:
    raise ValueError(
        "Cannot find predicted_labels column in adata.obs. "
        f"Available columns: {list(ad.obs.columns)[:50]} ..."
    )

epi_label = "Epithelial cells"
if epi_label not in ad.obs[label_col].astype(str).unique():
    top = ad.obs[label_col].astype(str).value_counts().head(30)
    raise ValueError(
        f"'{epi_label}' not found in obs['{label_col}'].\nTop labels:\n{top}"
    )

ad_epi = ad[ad.obs[label_col].astype(str) == epi_label].copy()
print(f"Epithelial cells: {ad_epi.n_obs:,} | genes: {ad_epi.n_vars:,}")

subset_path = os.path.join(OUTDIR, "adata_epithelial_subset.h5ad")
ad_epi.write(subset_path)
print(f"Saved subset: {subset_path}")

# ----------------------------
# Response column: values are "Responder"/"Non-Responder"
# ----------------------------
non_resp = {"PHD001", "PHD002", "PHD008"}
resp = {"PHD003", "PHD004", "PHD009"}
mapping = {p: "Non-Responder" for p in non_resp}
mapping.update({p: "Responder" for p in resp})

ad_epi.obs["Respond"] = ad_epi.obs["patient"].astype(str).map(mapping).fillna("Unknown")

rnr_col = pick_obs_col(
    ad_epi,
    ["response", "Respond", "responder", "Responder", "responder_status",
     "clinical_response", "group", "Group", "status", "RNR", "rnr"]
)
if rnr_col is None:
    raise ValueError(
        "Cannot find response column in adata.obs. "
        f"Available obs columns:\n{list(ad_epi.obs.columns)}"
    )

vals = ad_epi.obs[rnr_col].astype(str)

map_dict = {
    "Responder": "R",
    "Non-Responder": "NR",
    "responder": "R",
    "non-responder": "NR",
    "Non-responder": "NR",
    "NON-RESPONDER": "NR",
    "RESPONDER": "R",
}
ad_epi.obs["RNR_simplified"] = vals.map(map_dict).fillna(vals)

bad = set(ad_epi.obs["RNR_simplified"].unique()) - {"R", "NR"}
if len(bad) > 0:
    raise ValueError(
        f"Found unexpected response labels after mapping: {bad}\n"
        f"Original unique labels in '{rnr_col}': {sorted(vals.unique())[:50]}"
    )

# ----------------------------
# Clustering on X_scvi
# ----------------------------
print("[3/6] Neighbors/UMAP/Leiden on scVI latent (resolution=0.6)")
rep = pick_scvi_rep(ad_epi)
if rep is None:
    raise ValueError(f"Cannot find scVI latent in adata.obsm. Keys: {list(ad_epi.obsm.keys())}")
print(f"Using latent rep: {rep}")

sc.pp.neighbors(ad_epi, use_rep=rep, n_neighbors=15, random_state=0)
sc.tl.umap(ad_epi, random_state=0)
sc.tl.leiden(ad_epi, resolution=0.6, key_added="leiden_r0.6")

clustered_path = os.path.join(OUTDIR, "adata_epithelial_leiden_r0.6.h5ad")
ad_epi.write(clustered_path)
print(f"Saved clustered adata: {clustered_path}")

# ----------------------------
# UMAP plots
# ----------------------------
print("[4/6] Plotting UMAPs")
sc.pl.umap(ad_epi, color=["leiden_r0.6"], show=False)
save_current_fig(
    os.path.join(FIGDIR, "umap_epithelial_leiden_r0.6.png"),
    os.path.join(FIGDIR, "umap_epithelial_leiden_r0.6.pdf"),
)

sc.pl.umap(ad_epi, color=["RNR_simplified"], show=False)
save_current_fig(
    os.path.join(FIGDIR, "umap_epithelial_RNR.png"),
    os.path.join(FIGDIR, "umap_epithelial_RNR.pdf"),
)

# ----------------------------
# NR vs R counts per cluster (absolute)
# ----------------------------
print("[5/6] Computing NR vs R counts per cluster + barplot (absolute counts)")
ct = pd.crosstab(ad_epi.obs["leiden_r0.6"], ad_epi.obs["RNR_simplified"]).sort_index()
ct_path = os.path.join(TABDIR, "cluster_RNR_counts_absolute.csv")
ct.to_csv(ct_path)
print(f"Saved counts table: {ct_path}")

ax = ct.plot(kind="bar", stacked=True, figsize=(10, 4))
ax.set_xlabel("Epithelial Leiden cluster (res=0.6)")
ax.set_ylabel("Number of cells")
ax.set_title("Epithelial clusters: NR vs R absolute cell counts")
ax.legend(title="R/NR", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
save_current_fig(
    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_stacked.png"),
    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_stacked.pdf"),
)

ax = ct.plot(kind="bar", stacked=False, figsize=(10, 4))
ax.set_xlabel("Epithelial Leiden cluster (res=0.6)")
ax.set_ylabel("Number of cells")
ax.set_title("Epithelial clusters: NR vs R absolute cell counts")
ax.legend(title="R/NR", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
save_current_fig(
    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_grouped.png"),
    os.path.join(FIGDIR, "barplot_cluster_RNR_absolute_grouped.pdf"),
)

# ----------------------------
# DE: cluster vs rest using norm_expr_log1p
# ----------------------------
print("[6/6] DE (leiden cluster vs rest) using layer norm_expr_log1p")


X_backup = ad_epi.X
ad_epi.X = np.log1p(ad_epi.layers["norm_expr"])

sc.tl.rank_genes_groups(
    ad_epi,
    groupby="leiden_r0.6",
    method="wilcoxon",
    n_genes=200,
    use_raw=False
)

ad_epi.X = X_backup

de_df = sc.get.rank_genes_groups_df(ad_epi, group=None)
full_de_path = os.path.join(DEDIR, "rank_genes_groups_wilcoxon_all_clusters_top200.csv")
de_df.to_csv(full_de_path, index=False)
print(f"Saved full DE: {full_de_path}")

topN = 100
clusters = sorted(ad_epi.obs["leiden_r0.6"].unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))
sig_index_rows = []
for cl in clusters:
    df_cl = sc.get.rank_genes_groups_df(ad_epi, group=cl).dropna(subset=["names"]).copy()
    df_cl["cluster"] = cl
    out_path = os.path.join(SIGDIR, f"cluster_{cl}_signature_top{topN}.csv")
    df_cl.head(topN).to_csv(out_path, index=False)
    sig_index_rows.append((cl, out_path, df_cl.shape[0]))

sig_index = pd.DataFrame(sig_index_rows, columns=["cluster", "signature_file", "n_de_genes_returned"])
sig_index_path = os.path.join(SIGDIR, f"signature_files_index_top{topN}.csv")
sig_index.to_csv(sig_index_path, index=False)
print(f"Saved signature index: {sig_index_path}")

print("DONE.")
print(f"Outputs written to: {OUTDIR}")
